# Asari & Zador (2009) on CRCNS ac-1

Paper: H. Asari and A. M. Zador (2009), "Long-Lasting Context Dependence Constrains Neural Encoding
Models in Rodent Auditory Cortex", J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009.

Dataset: CRCNS ac-1 (`crcns:ac-1/unversioned`, doi:10.6080/K0KW5CXR), read from the public bucket
crcnsarchive. CRCNS data sets are not versioned, so the version pinned is the one the bucket serves;
its `checksums.md5` is printed below for the three zip files read, so that a later change can be
detected. There is no other version to choose from (docs/archives.md, CRCNS).

Claims: `claims/asari-2009.yaml`, written from the version of record (text extraction of the
publisher's PDF); the online supplement was not supplied to the readers.

The data set ships the authors' MATLAB analysis code (`asari/Tools`: `CxtDep.m`, `GetEvoked.m`,
`GetSpikes.m`). The paper does not link it, but it is the authors' code for this analysis, and the
helper `asari.py` follows it where the Methods leave a detail open; each such place is marked in
the helper.

| Claim | Tested here |
|---|---|
| methods-recording-counts | tested (voltage-clamp counts not testable) |
| a1-mgb-firing-rates | tested |
| fig2-example-reliability | tested |
| fig2-example-long-context | tested |
| fig3a-proportion-significant | tested |
| fig3a-significance-decay | tested |
| fig3b-fractional-power-decay | tested |
| timescale-longer-than-membrane | tested |
| fig4-predictability-upper-bound | tested |
| fig4-strf-performance-low | tested (16 of Wehr's 25 cells have repeated natural sounds) |
| fig5-example-conductances | blocked: data missing (no voltage-clamp recordings) |
| fig6-conductance-decay | blocked: data missing (no voltage-clamp recordings) |
| fig7-example-frequency-context | blocked: data missing (no cell with the example's three DMR bands) |
| fig8a-proportion-by-property | tested |
| fig8b-decay-by-property | tested |
| fig8a-significance-fits | tested |
| fig8c-relative-effects | tested |
| fig8-natural-synthetic-probes-alike | tested |
| fig9-mgb-fast-decay | tested |
| fig9-example-first-bin | tested |
| figs1-silence-probe | tested |

Wehr's recordings of Machens et al. (2004), used for fig4-strf-performance-low, are in the same
zip file (`wehr/`) and are read with the helper `wehr.py`.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fsspec
import asari as A

DATASET = "crcns:ac-1/unversioned"
T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

## Data check

The bucket holds three zip files: `crcns-ac1.zip` (Wehr's data, Asari's stimuli and tools, and
`asari/Summary.mat`, which lists the cells and their files) and two zips of Asari's result files.
Every result file is opened here and its metadata read: recording type, unit, sampling rate,
length and trigger types.

In [ ]:
print(fsspec.open(A.BUCKET + "checksums.md5", "r").open().read())
rows = pd.DataFrame(A.summary())
print(rows.groupby(["experiment", "exp_description"]).agg(cells=("cell", "nunique"), files=("file", "size")))
survey = pd.DataFrame(A.run_threads(A.survey_file, sorted(A.members())))
print("result files:", len(survey), " GB:", round(survey["size"].sum() / 1e9, 2))
print("files listed in Summary.mat but missing from the zips:", sorted(set(rows.file) - set(survey.path)))
print("files in the zips not listed in Summary.mat:", sorted(set(survey.path) - set(rows.file)))
print(survey.groupby(["rectype", "unit", "sf"]).size())
types = collections.Counter()
for t in survey.trigger_types:
    types.update(t)
print("trigger types over all files:", dict(types))
print("files with 'pulse' triggers:",
      survey[survey.trigger_types.map(lambda t: "pulse" in t)][["path", "rectype", "unit", "description"]].to_string())

All 346 result files store a trace in mV at 10 kHz, recorded in whole-cell (current clamp) or
cell-attached mode. No file holds currents, and no experiment in `Summary.mat` is a voltage-clamp
one: the paper's 42 voltage-clamped cells (Figures 5 and 6) are not in the data set. 'Pulse'
triggers appear in two current-clamp files only.

The analysis sets follow `Summary.mat`: experiment 0 ("Auditory cortical cell responses to natural
sound combinations", 39 cells) for the natural sound current-clamp data, experiment 1 ("Thalamic
cell responses", 14 cells) for MGB, experiment 2 (synthetic sounds, 96 cells) for the synthetic
ensembles. Tuning-curve files are left out of the context analyses.

In [ ]:
nat_files = rows[(rows.experiment == 0) & (rows.description != "tuning curve")].file.tolist()
mgb_files = rows[(rows.experiment == 1) & (rows.description != "tuning curve")].file.tolist()
syn_files = rows[(rows.experiment == 2) & (rows.description != "tuning curve")].file.tolist()
print("natural A1 files:", len(nat_files), " MGB files:", len(mgb_files), " synthetic A1 files:", len(syn_files))
sv = survey.set_index("path")
for name, fl in [("natural", nat_files), ("MGB", mgb_files), ("synthetic", syn_files)]:
    s = sv.loc[fl]
    print(name, dict(s.rectype.value_counts()), "units", set(s.unit), "sf", set(s.sf),
          "minutes per file %.1f to %.1f" % (s.n.min() / 6e5, s.n.max() / 6e5),
          "animals", len({A.animal_id(f) for f in fl}), "cells", len({A.cell_id(f) for f in fl}))
rec_x = A.Recording("Results/20051107-ha1-001/20051107-ha1-001-003.mat")
print("20051107-ha1-001-003: table rows", rec_x.table, "; trigger labels", sorted({t["description"] for t in rec_x.triggers if t["type"] != "pulse"}))
dup = sv.loc[nat_files + mgb_files + syn_files].groupby(["n", "size"]).size()
print("file pairs with identical length and size (possible double deposits):", int((dup > 1).sum()))

### Running the analyses

Each natural sound file is analysed in a worker process (`A.analyse_a1_file`): the trace is median
filtered (10 ms), cut into probe sets as `CxtDep.m` does, and for each probe the Kruskal-Wallis
significance measure (1,000 shuffles for the noise floor), the power terms of Eqs. 17 and 18, the
trial-to-trial correlations and the noise autocorrelation are computed, with all contexts and
without "silence". Spikes are counted by period. MGB files go through `A.analyse_mgb_file`
(spikes, PSTHs). Synthetic files go through `A.analyse_syn_file`: each probe is tested with the
variants of one property as its only contexts (see fig8a-proportion-by-property). This takes about
ten minutes.

In [ ]:
t = time.time()
NAT = A.run_parallel(A.analyse_a1_file, nat_files)
print("natural files analysed in %.0f s" % (time.time() - t))
t = time.time()
MGB = A.run_parallel(A.analyse_mgb_file, mgb_files)
print("MGB files analysed in %.0f s" % (time.time() - t))
t = time.time()
SYNR = A.run_parallel(A.analyse_syn_file, syn_files)
print("synthetic files analysed in %.0f s" % (time.time() - t))
print("peak memory of this process: %.2f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
# trial table checks: repeats, fragments, and the trigger clock
chk = pd.DataFrame([dict(file=A.cell_id(f["path"]) + "/" + f["path"].split("/")[-1][-7:-4], rep=f["rep"],
                         fragments=f["n_frag"], minutes=round(f["n_samples"] / f["sf"] / 60, 1),
                         vm_p1=round(f["vrange"][0], 1), vm_p99=round(f["vrange"][1], 1),
                         corr_aligned=round(f["clock"][0], 3), corr_shifted=round(f["clock"][1], 3))
                    for f in NAT])
print(chk.to_string())
print("aligned > shifted in %d of %d files; median aligned %.3f, shifted %.3f"
      % ((chk.corr_aligned > chk.corr_shifted).sum(), len(chk), chk.corr_aligned.median(), chk.corr_shifted.median()))
mg = pd.DataFrame([dict(file=A.cell_id(g["path"]) + "/" + g["path"].split("/")[-1][-7:-4], rep=g["rep"],
                        fragments=g["n_frag"], onsets=g["clock"][2], spikes_100ms_after_onset=g["clock"][0],
                        spikes_100ms_before_onset=g["clock"][1]) for g in MGB])
print(mg.to_string())
more = mg.spikes_100ms_after_onset > mg.spikes_100ms_before_onset
print("MGB files with more spikes after onsets than before: %d of %d; not: %s" % (more.sum(), len(mg), mg.file[~more].tolist()))

The median-filtered membrane potential has 1st percentiles from -84 to -43 mV and 99th
percentiles from -54 to -4 mV over the files, values expected for whole-cell recordings in mV. The correlation of the responses to the first sequence across repeats
is higher when the repeats are aligned on their triggers than when each is shifted by a different
lag, in the files listed above: the responses are locked to the trigger times, so the traces and
the trigger table share a clock. In MGB, spikes are more frequent in the 100 ms after stimulus
onsets than in the 100 ms before in 18 of 21 files, consistent with the triggers marking sound
onsets. In the other three (cells 20071012-ha1-002 and 20071012-ha1-003) there are fewer spikes
after onsets than before, so this contrast does not establish the clock for those two cells; their
probes are kept in the MGB analyses.

## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with natural
> sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated colored
> noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

A probe counts when it has at least two conditioning stimuli each with at least 4 trials (current
clamp) or 10 trials (MGB) (Methods, Context dependence at subthreshold voltage level; Table 1
legend). Readings: silence counted as a conditioning stimulus, or not. Synthetic files are cut into
probe sets in the same way (all contexts) for the cell count.

In [ ]:
def nat_probes(key="all"):
    out = []
    for f in NAT:
        for p in f["probes"]:
            if p["probe"] > 0 and p["m"] >= 4 and (key == "all" or p["ncx_nosil"] >= 2):
                out.append((f, p))
    return out

for key in ["all", "nosil"]:
    P = nat_probes(key)
    print("natural, silence %s: %d probes in %d cells, %d animals" % (
        "counted" if key == "all" else "not counted", len(P), len({A.cell_id(f["path"]) for f, p in P}),
        len({A.animal_id(f["path"]) for f, p in P})))
print("trials per context (m) over natural probes:", dict(collections.Counter(p["m"] for f, p in nat_probes())))
print("natural probes per file design:", dict(collections.Counter(f["n_frag"] for f, p in nat_probes())))
for key, rule in [("all", 10)]:
    P = [(g, p) for g in MGB for p in g["probes"] if p["m"] >= rule]
    print("MGB: %d probes with >= 10 trials per context in %d cells (%d probes in all files); trials per context %s"
          % (len(P), len({A.cell_id(g["path"]) for g, p in P}), sum(len(g["probes"]) for g in MGB),
             dict(collections.Counter(p["m"] for g, p in P))))
    P0 = [(g, p) for g, p in P if len([c for c in p["contexts"] if c != 0]) >= 2]
    print("MGB, silence not counted: %d probes in %d cells" % (len(P0), len({A.cell_id(g["path"]) for g, p in P0})))

def syn_count(path):
    rec = A.Recording(path)
    ps = A.probe_sets(rec, rec.trace)
    return dict(path=path, description=rec.description, rep=rec.rep,
                stimuli=sorted({s["description"][:24] for s in rec.stimuli}),
                probes_m4=sum(q["m"] >= 4 for q in ps),
                probes_m4_nosil=sum(q["m"] >= 4 and len([c for c in q["contexts"] if c != 0]) >= 2 for q in ps))
SYN = pd.DataFrame(A.run_threads(syn_count, syn_files))
SYN["cell"] = SYN.path.map(A.cell_id)
print(SYN.groupby("description").agg(files=("path", "size"), cells=("cell", "nunique"), probes=("probes_m4", "sum")).to_string())
syn_cells = set(SYN[SYN.probes_m4 > 0].cell)
nat_cells = {A.cell_id(f["path"]) for f, p in nat_probes()}
print("synthetic cells with a probe meeting the criterion: %d of %d; cells also in the natural set: %d"
      % (len(syn_cells), SYN.cell.nunique(), len(syn_cells & nat_cells)))
print("A1 current-clamp cells meeting the criterion (natural or synthetic): %d" % len(syn_cells | nat_cells))
print("excluding the files of 'naturalistic' sounds and of a natural probe after natural and pink-noise contexts "
      "(no Table 1 column): %d" % len(nat_cells | set(SYN[(SYN.probes_m4 > 0) & ~SYN.description.str.contains(
          "naturalistic|6 natural sound conditioning")].cell)))
print("synthetic probes with >= 4 trials per context, silence not counted: %d (silence counted: %d)"
      % (SYN.probes_m4_nosil.sum(), SYN.probes_m4.sum()))

# Table 1: probes of each probe type (rows) for each property varied among the conditioning
# stimuli (columns), with only the variants of that property as contexts (A.ensembles_for_probe;
# the map from stimulus parameters to properties is in A.PROPERTY_OF and its docstring)
ROW = {"natural": "Natural", "mcn": "MCN", "TORC": "TORC", "DMR": "DMR", "MHT": "MHT"}
def ens_family(f, p):
    kinds = {f["attrs"][c]["kind"] for c in p["contexts"]} | {p["kind"]}
    for k, name in [("mcn", "MCN"), ("TORC", "TORC"), ("DMR", "DMR"), ("MHT", "MHT")]:
        if k in kinds:
            return name
    return "natural variants"
ENS = pd.DataFrame([dict(file=f["path"], cell=A.cell_id(f["path"]), animal=A.animal_id(f["path"]),
                         description=f["description"], row=ROW[p["kind"]], prop=p["property"],
                         family=ens_family(f, p), probe=p["probe"], contexts=len(p["contexts"]), m=p["m"],
                         T=p["T"], tested=("res" in p), sig=("res" in p and len(p["res"]["runs"]) > 0))
                    for f in SYNR for p in f["probes"]])
TABLE1 = {("Natural", "amplitude"): (23, 23), ("Natural", "frequency"): (25, 25), ("Natural", "AM"): (25, 25),
          ("Natural", "FM"): (25, 25), ("Natural", "higher-order"): (63, 27), ("TORC", "amplitude"): (39, 9),
          ("TORC", "frequency"): (40, 9), ("DMR", "amplitude"): (8, 2), ("DMR", "frequency"): (20, 5),
          ("MHT", "amplitude"): (25, 20), ("MHT", "frequency"): (25, 21), ("MHT", "AM"): (71, 27),
          ("MHT", "FM"): (57, 25), ("MCN", "higher-order"): (74, 27)}
ok4 = ENS[ENS.tested]
t1 = ok4.groupby(["row", "prop"]).agg(probes=("sig", "size"), cells=("cell", "nunique"))
t1 = t1.reindex(sorted(set(t1.index) | set(TABLE1))).fillna(0).astype(int)
t1["paper_probes"] = [TABLE1.get(i, (0, 0))[0] for i in t1.index]
t1["paper_cells"] = [TABLE1.get(i, (0, 0))[1] for i in t1.index]
print("Table 1, probes (cells) with >= 4 trials in each of >= 2 variant contexts:")
print(t1.to_string())
print("column totals (data / paper):", {P: (int(ok4[ok4.prop == P].shape[0]),
      sum(v[0] for k, v in TABLE1.items() if k[1] == P)) for P in A.PROPERTIES})
print("ensembles with fewer than 4 trials in some context (not counted):",
      ENS[~ENS.tested].groupby(["row", "prop"]).size().to_dict())
fam = ok4.groupby("family").cell.nunique()
print("cells by synthetic ensemble used (data / paper: TORC 14, DMR 6, MHT 39, MCN 27):", fam.to_dict())
print("cells with a Table 1 probe, natural sound ensembles included: %d"
      % len(set(ok4.cell) | nat_cells))

**Result: partially reproduced; the natural sound and MGB counts match the paper exactly, the 123 cells do not, and the voltage-clamp counts cannot be tested.** There are 305 natural sound probes in 39 A1 cells whether or not silence counts as a conditioning stimulus. There are 93 MGB probes in 14 cells. 132 A1 current-clamp cells meet the criterion (39 natural, 95 synthetic, 2 in both), against the paper's 123; leaving out the two synthetic designs named in the output gives 124, and 132 cells have a probe that enters a Table 1 column. In Table 1, the column totals for amplitude (95 probes) and frequency (110) equal the paper's, and the DMR rows (8 and 20 probes in 2 and 5 cells), the natural AM and FM entries (25 probes each) and the TORC cell counts (9 and 9) match. Other entries differ: there are no modulated harmonic tone probes for amplitude, frequency or FM, and 14 for AM against 71, so the AM and FM totals are 39 and 25 against 96 and 82; the natural amplitude and frequency entries have 45 and 46 probes against 23 and 25. By the synthetic ensemble used, there are 14 TORC cells (paper 14), 15 DMR (6), 26 MHT (39) and 26 MCN (27). The voltage-clamp counts (29 probes, 14 cells) cannot be tested, as the data set holds no voltage-clamp recordings.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD,
> 194 cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

Rates per cell over its context files; spontaneous over the intersequence silences or the 1 s
before each sequence (baseline), evoked over the sound periods. A1 spikes are excursions of the raw
trace above its median-clipped trace (5, 10 or 20 mV, an assumption); MGB spikes on the code's
high-pass filtered trace, with the code's automatic threshold or 20 or 40 robust SDs. The data set
holds 39 natural sound A1 cells and 14 MGB cells with context files, not the paper's 194 and 16.
The probe periods analysed cover every sound fragment, so the second evoked reading equals the first.

In [ ]:
cell_of = {f["path"]: A.cell_id(f["path"]) for f in NAT}
def cell_rates(files, key, spont):
    acc = {}
    for f in files:
        r = f["rates"][key]
        a = acc.setdefault(A.cell_id(f["path"]), np.zeros(4))
        a += [r[spont][0], r[spont][1], r["sound"][0], r["sound"][1]]
    s = np.array([v[0] / v[1] for v in acc.values()])
    e = np.array([v[2] / v[3] for v in acc.values()])
    return len(acc), s, e

expect = {"A1": (0.47, 0.61, 0.57, 0.77), "MGB": (0.78, 1.25, 11.4, 16.9)}
tab = []
for area, files, keys in [("A1", NAT, [5.0, 10.0, 20.0]), ("MGB", MGB, list(A.MGB_THRESHOLDS))]:
    for key in keys:
        for spont in ["silence", "baseline"]:
            n, s, e = cell_rates(files, key, spont)
            ex = expect[area]
            tab.append(dict(area=area, threshold=key, spontaneous_window=spont, cells=n,
                            spont_mean=s.mean(), spont_sd=s.std(ddof=1), evoked_mean=e.mean(), evoked_sd=e.std(ddof=1),
                            spont_ok=abs(s.mean() - ex[0]) <= 0.1 * ex[0], evoked_ok=abs(e.mean() - ex[2]) <= 0.1 * ex[2],
                            spont_max=s.max(), evoked_max=e.max()))
tab = pd.DataFrame(tab)
print(tab.round(3).to_string())
a1ok = tab[tab.area == "A1"]
mgok = tab[tab.area == "MGB"]
print("A1 readings with both means within 10%:", int((a1ok.spont_ok & a1ok.evoked_ok).sum()), "of", len(a1ok))
print("MGB readings with both means within 10%:", int((mgok.spont_ok & mgok.evoked_ok).sum()), "of", len(mgok))
print("A1 means below 1 Hz in every reading:", bool((a1ok[["spont_mean", "evoked_mean"]] < 1).all().all()))
print("MGB evoked mean above 5 Hz:", dict(zip(mgok.threshold.astype(str) + "/" + mgok.spontaneous_window, (mgok.evoked_mean > 5))))
print("MGB spike thresholds (mV on the filtered trace), median over files:",
      {k: round(float(np.median([g["threshold"][k] for g in MGB])), 3) for k in A.MGB_THRESHOLDS})

In [ ]:
# all A1 current-clamp cells (natural and synthetic files), and checks of each spike detector
allfiles = NAT + SYNR
rows_all = []
for key in [5.0, 10.0, 20.0]:
    for spont in ["silence", "baseline"]:
        n, s, e = cell_rates(allfiles, key, spont)
        rows_all.append(dict(threshold=key, spontaneous_window=spont, cells=n, spont_mean=s.mean(), spont_sd=s.std(ddof=1),
                             evoked_mean=e.mean(), evoked_sd=e.std(ddof=1),
                             spont_ok=abs(s.mean() - 0.47) <= 0.047, evoked_ok=abs(e.mean() - 0.57) <= 0.057))
print("A1, all current-clamp cells with context files (natural and synthetic):")
print(pd.DataFrame(rows_all).round(3).to_string())
chk_rows = []
for area, files, keys in [("A1", NAT, [5.0, 10.0, 20.0]), ("MGB", MGB, list(A.MGB_THRESHOLDS))]:
    for key in keys:
        sc = [f["spike_check"][key] for f in files]
        n = np.array([c["n"] for c in sc])
        chk_rows.append(dict(area=area, threshold=key, files=len(sc), spikes=int(n.sum()), files_without_spikes=int((n == 0).sum()),
                             spikes_per_file_median=float(np.median(n)), spikes_per_file_max=int(n.max()),
                             short_isi_fraction_median=float(np.nanmedian([c["short_isi"] for c in sc])),
                             short_isi_fraction_max=float(np.nanmax([c["short_isi"] for c in sc]))))
print("spike detectors: interspike intervals under 1.5 ms (refractory violations) and spikes per file")
print(pd.DataFrame(chk_rows).round(4).to_string())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for j, (area, files, keys) in enumerate([("A1", NAT, [5.0, 10.0, 20.0]), ("MGB", MGB, list(A.MGB_THRESHOLDS))]):
    for key in keys:
        W = [f["spike_check"][key]["wave_mean"] for f in files if f["spike_check"][key]["wave_mean"] is not None]
        w = np.mean(W, axis=0)
        ax[j].plot((np.arange(len(w)) - len(w) // 2) / 10, w, label=str(key))
        print("%s %s: mean waveform over files, peak %.2f mV at 0 ms, value at -2 and +2 ms %.2f and %.2f mV"
              % (area, key, w[len(w) // 2], w[0], w[-1]))
    ax[j].set_title(area + (" (raw minus median-clipped trace)" if area == "A1" else " (high-pass filtered)"))
    ax[j].set_xlabel("time from spike (ms)"); ax[j].legend(title="threshold")
mgb_dead = [dict(path=g["path"], rates=g["rates_dead"]) for g in MGB]
rows_dead = []
for key in A.MGB_THRESHOLDS:
    for spont in ["silence", "baseline"]:
        n, s, e = cell_rates(mgb_dead, key, spont)
        rows_dead.append(dict(threshold=key, spontaneous_window=spont, cells=n, spont_mean=s.mean(), spont_sd=s.std(ddof=1),
                              evoked_mean=e.mean(), evoked_sd=e.std(ddof=1), spont_ok=abs(s.mean() - 0.78) <= 0.078,
                              evoked_ok=abs(e.mean() - 11.4) <= 1.14, evoked_above_5=e.mean() > 5,
                              spikes=sum(g["spike_check_dead"][key]["n"] for g in MGB),
                              spikes_without_dead_time=sum(g["spike_check"][key]["n"] for g in MGB)))
print("MGB with a 1.5-ms dead time after each spike:")
print(pd.DataFrame(rows_dead).round(3).to_string())


**Result: partially reproduced; no reading puts all four means within 10%, but A1 rates are below 1 Hz and MGB evoked rates are well above them.** No A1 reading has both means within tolerance, over the 39 natural sound cells or over all 133 current-clamp cells with context files: with a 10 mV threshold the evoked mean matches (0.625 and 0.608 Hz) but the spontaneous mean is higher, and with 20 mV both are lower. No MGB reading has both. A1 means are below 1 Hz in every reading. The A1 detectors give almost no interspike intervals under 1.5 ms (at most 0.6% in a file), and their mean waveform is a spike of 30 to 39 mV. The MGB evoked mean is above 5 Hz with the 20 and 40 SD thresholds, but not with the code's automatic threshold (5.0 Hz), which is set 5 to 9 times higher. The 20 SD threshold splits spikes (a median of 23% of intervals under 1.5 ms), and at 40 SD some files do too (up to 80%). With a 1.5-ms dead time after each spike the 40 SD threshold gives 0.78 Hz spontaneous (within tolerance) and 8.60 Hz evoked, and 20 SD gives 9.62 Hz evoked, so the MGB evoked mean stays above 5 Hz; with the automatic threshold it is 4.97 Hz.

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (correlation across trials
> 0.74 +/- 0.08 for the shown probe and 0.61 +/- 0.07 over the seven natural sound fragments tested),
> and the mean correlation over the population was 0.31 +/- 0.09.

Readings: (a) mean correlation over pairs of trials, (b) each trial against the mean of the others;
population mean over cells, or over probes. Correlations are over the probe period, within each
context, then averaged over contexts.

In [ ]:
P = nat_probes()
df = pd.DataFrame([dict(cell=A.cell_id(f["path"]), animal=A.animal_id(f["path"]), file=f["path"], probe=p["probe"],
                        a=p["corr"][0], b=p["corr"][1], n_frag=f["n_frag"], rep=f["rep"], contexts=p["contexts"])
                   for f, p in P])
for r in ["a", "b"]:
    per_cell = df.groupby("cell")[r].mean()
    print("reading %s: over probes %.3f +/- %.3f (n=%d); over cells %.3f +/- %.3f (n=%d)"
          % (r, df[r].mean(), df[r].std(ddof=1), len(df), per_cell.mean(), per_cell.std(ddof=1), len(per_cell)))
print("range over cells, reading a: %.2f to %.2f" % (df.groupby("cell").a.mean().min(), df.groupby("cell").a.mean().max()))
cand = df[(df.n_frag == 7) & (df.rep == 6)]
print("cells with seven natural fragments and six repeats:")
print(cand.groupby("cell").agg(a=("a", "mean"), a_sd=("a", "std"), b=("b", "mean"), probes=("probe", "size")).round(3))
print(cand[["cell", "probe", "contexts", "a", "b"]].round(3).to_string())

**Result: reproduced; the population mean is 0.31 under reading (a), and cell 20051222-ha1-001 matches the paper's example values.** Under reading (a), the mean correlation over cells is 0.31 (SD 0.09) and over probes 0.32. Reading (b) gives higher values, outside the tolerance. Of the five cells with seven natural fragments and six repeats, 20051222-ha1-001 has a mean of 0.61 over its seven fragments, and its probe 4 (contexts silence, 2 and 3) has 0.74. These are the paper's example values, so it is taken as the example cell.

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe for
> about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again after
> about 4 s.

The claim was amended after a question to a reader (`questions/asari-2009.yaml`). The example is
matched by the reliability values of the Figure 2A legend: a current-clamp cell with seven natural
sound fragments whose mean reliability (reading (a) of fig2-example-reliability) is within
0.61 +/- 0.07, and in it a probe tested after silence and two other fragments, with six repeats in at
least one context, whose reliability is within 0.74 +/- 0.08; the closest to both values where
several match. Reproduced when the matched probe has significant periods before 2 s and after
3.6 s and none between 2 and 3.6 s. The strict reading (none between 2 and 4 s) is reported, and
every probe with seven fragments, silence and two fragments is shown.


In [ ]:
def runs_s(p, key="all"):
    return [(round(a / 1e4, 3), round(b / 1e4, 3)) for a, b in p[key]["runs"]]
# matching rule of the amended criterion (Figure 2A legend values)
cell_rel = df[df.n_frag == 7].groupby("cell").a.mean()
print("cells with seven natural fragments, mean reliability (reading a):")
print(cell_rel.round(3).to_string())
match = []
for f, p in P:
    cid = A.cell_id(f["path"])
    if f["n_frag"] != 7 or abs(cell_rel[cid] - 0.61) > 0.07:
        continue
    ok_ctx = len(p["contexts"]) == 3 and 0 in p["contexts"]
    six = max(p["n_occ"].values()) >= 6
    match.append(dict(cell=cid, cell_rel=round(cell_rel[cid], 3), probe=p["probe"], contexts=p["contexts"],
                      six_repeats=six, probe_rel=round(p["corr"][0], 3),
                      matches=ok_ctx and six and abs(p["corr"][0] - 0.74) <= 0.08,
                      distance=round(abs(cell_rel[cid] - 0.61) + abs(p["corr"][0] - 0.74), 3)))
match = pd.DataFrame(match)
print("probes of cells within 0.61 +/- 0.07:")
print(match.to_string())
best = match[match.matches].sort_values("distance").iloc[0]
print("matched probes: %d; closest: %s probe %d" % (match.matches.sum(), best.cell, best.probe))
f_ex = [f for f, p in P if A.cell_id(f["path"]) == best.cell][0]
p_ex = [p for p in f_ex["probes"] if p["probe"] == best.probe][0]

def window_test(r, lo, hi):
    return dict(before_2s=any(a < 2 for a, b in r), after_hi=any(b > hi for a, b in r),
                none_in_window=not any(b > lo and a < hi for a, b in r),
                overlap_ms=round(1000 * sum(max(0, min(b, hi) - max(a, lo)) for a, b in r), 1))
r = runs_s(p_ex)
print("matched probe, significant periods (s), all contexts:", [(float(a), float(b)) for a, b in r])
print("main reading, 2 to 3.6 s:", window_test(r, 2.0, 3.6))
print("strict reading, 2 to 4 s:", window_test(r, 2.0, 4.0))
gap = [(float(b0), float(a1)) for (a0, b0), (a1, b1) in zip(r[:-1], r[1:]) if a1 - b0 > 1]
print("gaps longer than 1 s (s):", gap)
print("same probe without silence:", [(float(a), float(b)) for a, b in runs_s(p_ex, "nosil")])
rowsx = []
for f, p in P:
    if f["n_frag"] == 7 and len(p["contexts"]) == 3 and 0 in p["contexts"]:
        rr = runs_s(p)
        w36, w4 = window_test(rr, 2.0, 3.6), window_test(rr, 2.0, 4.0)
        rowsx.append(dict(cell=A.cell_id(f["path"]), probe=p["probe"], T_s=p["T"] / 1e4, n_periods=len(rr),
                          before_2s=w36["before_2s"], after_3_6s=w36["after_hi"], none_2_3_6=w36["none_in_window"],
                          after_4s=w4["after_hi"], none_2_4=w4["none_in_window"], overlap_2_4_ms=w4["overlap_ms"]))
print("all probes with seven fragments and contexts silence plus two fragments:")
print(pd.DataFrame(rowsx).to_string())
late = [(A.cell_id(f["path"]), p["probe"]) for f, p in P if any(b > 4e4 for a, b in p["all"]["runs"])]
print("natural probes with a significant period later than 4 s: %d of %d" % (len(late), len(P)))


**Result: reproduced under the amended criterion; the strict 2 to 4 s reading fails by 30 ms.** Only one probe matches the Figure 2A values: probe 4 of 20051222-ha1-001 (cell mean 0.611, probe 0.737). No other cell with seven fragments has a mean within 0.61 +/- 0.07. In this probe, context dependence is significant from 0.03 s to 1.60 s and again from 3.97 s to 4.60 s, with nothing significant between 1.60 and 3.97 s. So there are significant periods before 2 s and after 3.6 s, and none between 2 and 3.6 s. Under the strict reading, the period starting at 3.97 s overlaps the 2 to 4 s interval by 30 ms. Without silence among the contexts, the probe has only two short significant periods. 17 of 305 natural probes have a significant period later than 4 s.


## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound
> contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of the
> significant events occurred more than 1 s after probe onset.

Readings: silence counted among the contexts or not; the proportion at time t over all 305 probes,
or over the probes still running at t. The late fraction is one minus the cumulative probability at
1 s, the cumulative being the running integral of the proportion less the noise floor, normalised at
its peak (Figure 3A legend).

In [ ]:
def population(key, denom):
    PP = nat_probes(key)
    L = max(p["T"] for f, p in PP) // 10
    S, R, NZ = np.zeros(L), np.zeros(L), np.zeros(L)
    for f, p in PP:
        s = np.zeros(p["T"], bool)
        for a, b in p[key]["runs"]:
            s[a:b] = True
        sb = A.bin_mean(s.astype(float), 10)
        S[:len(sb)] += sb
        R[:len(sb)] += 1
        NZ[:len(p[key]["noise"])] += p[key]["noise"]
    den = len(PP) if denom == "all" else np.maximum(R, 1)
    return np.arange(L) * 1e-3, S / den, NZ / den, R, PP

FIT3A = {}
res = []
for key in ["all", "nosil"]:
    for denom in ["all", "running"]:
        t, prop, nf, R, PP = population(key, denom)
        nsig = sum(len(p[key]["runs"]) > 0 for f, p in PP)
        c = np.cumsum(prop - nf)
        ipk = int(np.argmax(c))
        cum = c / c[ipk]
        FIT3A[(key, denom)] = (t, prop, nf)
        res.append(dict(silence=("counted" if key == "all" else "not counted"), denominator=denom, probes=len(PP),
                        significant=nsig, fraction=nsig / len(PP), cum_peak_s=t[ipk], late_fraction=1 - cum[1000],
                        noise_floor_mean_0_6s=nf[t < 6].mean()))
res = pd.DataFrame(res)
res["count_ok"] = (res.significant - 204).abs() <= 20.4
res["fraction_ok"] = (res.fraction - 0.669).abs() <= 0.0669
res["late_ok"] = (res.late_fraction - 0.237).abs() <= 0.0237
print(res.round(4).to_string())
df_sig = pd.DataFrame([dict(animal=A.animal_id(f["path"]), cell=A.cell_id(f["path"]), sig=len(p["all"]["runs"]) > 0)
                       for f, p in nat_probes()])
by_animal = df_sig.groupby("animal").sig.agg(["sum", "size", "mean"])
print("by animal (silence counted): %d animals; fraction significant per animal median %.2f, range %.2f to %.2f"
      % (len(by_animal), by_animal["mean"].median(), by_animal["mean"].min(), by_animal["mean"].max()))
print(by_animal.T.to_string())
t, prop, nf = FIT3A[("all", "all")]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, prop, "k", lw=1, label="proportion significant")
ax.plot(t, nf, color="0.6", label="noise floor")
ax.set_xlim(0, 6); ax.set_xlabel("time from probe onset (s)"); ax.set_ylabel("probability"); ax.legend()
print("plotted range: proportion %.3f to %.3f over 0 to %.2f s; none outside the axes up to 6 s" % (prop.min(), prop.max(), t[-1]))

In [ ]:
tf = np.array([p["all"]["tie_frac"] for f, p in nat_probes()])
print("time points with tied values among a probe's trials: median %.3f of the points over 305 probes (range %.3f to %.3f)"
      % (np.median(tf), tf.min(), tf.max()))

**Result: reproduced; with silence counted and the proportion taken over all 305 probes, all three values are within tolerance.** 186 of 305 probes (61.0%) are significant, the late fraction is 0.252 and the cumulative curve peaks at 4.78 s (paper 4.72 s). The noise floor of this reading averages 2.2 x 10^-3, against the paper's 2.8 x 10^-3; over the probes still running it is 2.8 x 10^-3, but the late fraction there is 0.267, outside tolerance, so the count and the chance level are matched by different readings. With silence left out of the contexts, 166 probes (54.4%) are significant and the late fraction is 0.17 to 0.18, both outside tolerance. Tied values occur at a median of 31% of the time points of a probe, so the rank handling of ties matters (see the summary).

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the sum
> of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise floor of
> about 2.8 x 10^-3.

Eq. 44 with the constant fixed at the mean noise floor, fitted from 0 to 4.72 s or 0 to 6 s, with
the paper's rule for the number of exponentials. Sizes are fitted with free sign, or bounded at zero
(the paper calls them decay sizes); both are shown.

In [ ]:
out = []
for (key, denom), (t, prop, nf) in FIT3A.items():
    kappa = nf[t < 6].mean()
    for t_end in [4.72, 6.0]:
        sel = t <= t_end
        for nonneg in [False, True]:
            k, fits = A.fit_exp_select(t[sel], prop[sel], kappa=kappa, nonneg=nonneg)
            f2 = fits[2]
            out.append(dict(silence=key, denominator=denom, fit_to=t_end, sizes=("nonneg" if nonneg else "free"),
                            n_selected=k, kappa=kappa, a1=f2["a"][0], tau1=f2["tau"][0], a2=f2["a"][1], tau2=f2["tau"][1],
                            one_exp_a=fits[1]["a"][0], one_exp_tau=fits[1]["tau"][0]))
out = pd.DataFrame(out)
exp3a = dict(a1=(0.17, 0.017), tau1=(0.20, 0.02), a2=(0.09, 0.009), tau2=(0.90, 0.09))
for c, (v, tol) in exp3a.items():
    out[c + "_ok"] = (out[c] - v).abs() <= tol + 1e-12
out["kappa_ok"] = (out.kappa - 0.0028).abs() <= 0.00028
out["partial"] = (out.n_selected == 2) & out.tau2.between(0.5, 2)
print(out.round(4).to_string())

**Result: partially reproduced, resting on the assumption that sizes are non-negative; no reading puts every value within tolerance.** With sizes bounded at zero and silence counted, two exponentials are selected in every reading. The slow constant is 0.97 to 1.08 s, within tolerance in one reading (all probes, fit to 6 s). The fast constant is 0.25 to 0.27 s and the slow size 0.068 to 0.080, both outside tolerance. The fast size is 0.149 to 0.160, within tolerance when the proportion is over the probes still running. With free signs, the fit spends one exponential on the rise of the curve after probe onset (a negative size), and the slow constant is 0.44 s, below the 0.5 s of the partial rule. Without silence, one exponential is selected. The mean noise floor is 2.8 x 10^-3 over the probes still running, and 2.2 x 10^-3 over all probes.

## fig3b-fractional-power-decay and fig4-predictability-upper-bound

> fig3b: The context-dependent fraction of the stimulus-related response power, normalized by the
> smoothed stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

> fig4: Without knowledge of the context, no model can capture more than about half (0.51) of the
> predictable response power with a window of about 100 ms; the upper bound approaches one only
> with a window of seconds (time constant 1.04 s).

Eq. 17 (stimulus-related) and Eq. 18 (context-independent) are averaged over probes at each time
point; Q(t) of Eq. 20 is (Eq. 17 - Eq. 18) over the running mean of Eq. 17 over [0, 2t]; the upper
bound of Eq. 43 is Eq. 18 over the same smoothed denominator (main reading) or over the unsmoothed
one (variant). Readings: silence counted or not, average over all probes or over the probes
running, fit to 4 s or 6 s.

In [ ]:
POW = {}
out = []
for key in ["all", "nosil"]:
    PP = nat_probes(key)
    L = max(len(p[key]["pred"]) for f, p in PP)
    t = np.arange(L) * 1e-3
    PR, IV, R = np.zeros(L), np.zeros(L), np.zeros(L)
    for f, p in PP:
        n = len(p[key]["pred"])
        PR[:n] += p[key]["pred"]; IV[:n] += p[key]["inv"]; R[:n] += 1
    for denom in ["all", "running"]:
        den = len(PP) if denom == "all" else np.maximum(R, 1)
        pr, iv = PR / den, IV / den
        sm = A.running_mean_0_2t(pr, 1e-3)
        Q, U, U0 = (pr - iv) / sm, iv / sm, iv / pr
        POW[(key, denom)] = (t, pr, iv, sm, Q, U)
        for t_end in [4.0, 6.0]:
            sel = t <= t_end
            fq = A.fit_exp(t[sel], Q[sel], 1, kappa=0.0)
            fu = A.fit_exp(t[sel], U[sel], 1, kappa=1.0)
            f0 = A.fit_exp(t[sel], U0[sel], 1, kappa=1.0)
            out.append(dict(silence=key, denominator=denom, fit_to=t_end, Q_a1=fq["a"][0], Q_tau1=fq["tau"][0],
                            U_1plus_a1=1 + fu["a"][0], U_tau1=fu["tau"][0],
                            U_unsmoothed_1plus_a1=1 + f0["a"][0], U_unsmoothed_tau1=f0["tau"][0],
                            Q_at_0=Q[0], Q_at_1s=Q[1000], U_at_0_1s=U[100]))
out = pd.DataFrame(out)
out["fig3b_ok"] = ((out.Q_a1 - 0.49).abs() <= 0.049) & ((out.Q_tau1 - 1.04).abs() <= 0.104)
out["fig4_ok"] = ((out.U_1plus_a1 - 0.51).abs() <= 0.051) & ((out.U_tau1 - 1.04).abs() <= 0.104)
out["fig4_unsmoothed_ok"] = ((out.U_unsmoothed_1plus_a1 - 0.51).abs() <= 0.051) & ((out.U_unsmoothed_tau1 - 1.04).abs() <= 0.104)
print(out.round(3).to_string())
t, pr, iv, sm, Q, U = POW[("all", "all")]
print("stimulus-related power (mV^2) at 0, 0.1, 1, 4 s: %s; smoothed: %s" % (pr[[0, 100, 1000, 3999]].round(2), sm[[0, 100, 1000, 3999]].round(2)))
print("Q range over 0-6 s: %.3f to %.3f; U range: %.3f to %.3f" % (Q[t < 6].min(), Q[t < 6].max(), U[t < 6].min(), U[t < 6].max()))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
sel = t < 6
ax[0].plot(t[sel], Q[sel], "k", lw=0.8); ax[0].set_title("Q(t), Eq. 20"); ax[0].set_ylim(-0.2, 1.2)
ax[1].plot(t[sel], U[sel], "k", lw=0.8); ax[1].set_title("upper bound, Eq. 43"); ax[1].set_ylim(-0.2, 1.2)
for a_ in ax: a_.set_xlabel("time from probe onset (s)")
print("points outside the plotted range -0.2 to 1.2: Q %d, U %d" % (((Q[sel] < -0.2) | (Q[sel] > 1.2)).sum(), ((U[sel] < -0.2) | (U[sel] > 1.2)).sum()))

In [ ]:
# where the long time constant comes from, the 1 - Q reading of Figure 4, and fits with kappa free
t, pr, iv, sm, Q, U = POW[("all", "all")]
sel = t <= 4.0
fq = A.fit_exp(t[sel], Q[sel], 1, kappa=0.0)
fqk = A.fit_exp(t[sel], Q[sel], 1, kappa=None)
model = fq["a"][0] * np.exp(-t / fq["tau"][0])
modelk = fqk["a"][0] * np.exp(-t / fqk["tau"][0]) + fqk["kappa"]
paper = 0.49 * np.exp(-t / 1.04)
for tt in [0.5, 1, 2, 3, 4]:
    i = int(round(tt * 1000))
    print("t = %.1f s: Q %.3f (10-ms mean %.3f), fit %.3f, fit with kappa free %.3f, paper's curve 0.49 exp(-t/1.04) %.3f"
          % (tt, Q[i], Q[max(0, i - 5):i + 5].mean(), model[i], modelk[i], paper[i]))
for a, b in [(0, 1), (1, 2), (2, 4)]:
    s_ = (t >= a) & (t < b)
    print("mean residual of the fit (Q - fit) over %d to %d s: %.4f; RMS %.4f" % (a, b, (Q - model)[s_].mean(), np.sqrt(((Q - model)[s_] ** 2).mean())))
print("Q fit with kappa free (0 to 4 s): alpha1 %.3f, tau1 %.3f s, kappa %.3f" % (fqk["a"][0], fqk["tau"][0], fqk["kappa"]))
out4 = []
for (key, denom), (t_, pr_, iv_, sm_, Q_, U_) in POW.items():
    for t_end in [4.0, 6.0]:
        s_ = t_ <= t_end
        f1 = A.fit_exp(t_[s_], 1 - Q_[s_], 1, kappa=1.0)
        out4.append(dict(silence=key, denominator=denom, fit_to=t_end, one_minus_Q_1plus_a1=1 + f1["a"][0], one_minus_Q_tau1=f1["tau"][0]))
out4 = pd.DataFrame(out4)
out4["fig4_ok"] = ((out4.one_minus_Q_1plus_a1 - 0.51).abs() <= 0.051) & ((out4.one_minus_Q_tau1 - 1.04).abs() <= 0.104)
print("Figure 4 read as 1 - Q (the legend's alpha1 = -0.49 and tau1 = 1.04 s are those of Figure 3B):")
print(out4.round(3).to_string())
print("fits of the smoothed U that stop at the 100 s bound on tau:",
      out[out.U_tau1 >= 99][["silence", "denominator", "fit_to", "U_1plus_a1", "U_tau1"]].round(3).to_string())
print("smoothed U above 1.2 at %d of %d points over 0 to 6 s (all probes, silence counted); the smoothed denominator"
      " falls faster than the context-independent power late in the probe" % ((U[t < 6] > 1.2).sum(), (t < 6).sum()))

**Result: fig3b-fractional-power-decay is not reproduced; the size matches, but the time constant is longer than 1.04 s in every reading.** The size of Q(t) is 0.45 to 0.47, within tolerance. The time constant is 1.3 to 1.8 s. Q itself follows the paper's curve up to 1 s (0.294 and 0.191 at 0.5 and 1 s, against 0.303 and 0.187) and departs from it later: it is 0.053, 0.138 and 0.115 at 2, 3 and 4 s, against 0.072, 0.027 and 0.010. The single exponential with constant 0 is pulled up by this tail (mean residual 0.031 over 2 to 4 s); with the constant free the fit gives 0.57 s and a constant of 0.12.

**Result: fig4-predictability-upper-bound is not reproduced; the upper bound near onset is close to one half, but the time constant is longer than 1.04 s in every reading.** With the smoothed denominator, 1 + alpha1 is 0.43 to 0.49 in the fits that stay within bounds, within tolerance in three of them. Their time constant is 1.66 to 2.81 s; the two fits to 6 s over all probes run to the fit's 100 s bound, and the smoothed ratio exceeds 1.2 at 176 of 6000 points, so this reading is erratic late in the probe. With the unsmoothed denominator, 1 + alpha1 is 0.50 to 0.54, within tolerance in every reading, and the time constant is 1.46 to 1.92 s. Read as 1 - Q (the Figure 4 legend gives the values of Figure 3B), 1 + alpha1 is 0.53 to 0.56 and the time constant 1.29 to 1.78 s, again outside tolerance.

## fig4-strf-performance-low

> Linear STRF models of 20 A1 cells from Machens et al. (2004) predicted less than about 20% of the
> predictable response power, and extending the window from 180 ms to about 4 s did not improve them
> significantly, with or without a static nonlinearity; the LN model did only slightly better.

Wehr's recordings for Machens et al. (2004) are in `crcns-ac1.zip` under `wehr/`, one file per
trial, with the stimuli and Wehr's tools (`STRFanalysis.m`, `ridgesvd1.m`, `logspectrogram.m`). The
helper `wehr.py` follows the Methods (Neural encoding models; Model performance): spectrogram Eq. 47
at 3 bins per octave from 0.40 kHz (17 bins), time bins of 2^k ms for k = 2 to 10 split as the
Methods list (windows of 180 to 4052 ms, 765 parameters), ridge regression (Eq. 50) with 10-fold
cross-validation choosing the ridge value on the validation fold, LN by robust lowess (5% span, 5
iterations) on the training folds, and performance by Eq. 53 on training (upper) and validation
(lower) data. Eq. 53 needs at least two trials per stimulus, so cells whose natural sounds were all
presented once cannot be used. Readings: the spectrogram window (10-ms and 20-ms Hann), and how the
per-fold values are combined (the mean of the fold ratios, as the Methods and `STRFanalysis.m` do,
and the ratio of the means over folds, as a variant, since single folds with almost no predictable
power give unstable ratios). The response is sampled in 4-ms bins; the 10-kHz reading of the
criterion does not apply, as these traces are sampled at 4 kHz. Folds are contiguous blocks; the
by-stimulus reading was not run. The criterion's test decides "did not improve": a one-sided paired
Wilcoxon test over cells of validation performance at 4052 ms against 180 ms, at 0.05, for STRF and
LN; the Friedman test over the nine windows is a variant.

In [ ]:
import wehr as W
from scipy import stats
cells_w = W.summary()
for e in (0, 1):
    print("Wehr experiment %d: %d cells; %s" % (e, sum(c["exp"] == e for c in cells_w),
          " ".join(str([c["exp_description"] for c in cells_w if c["exp"] == e][0]).split())))
usable = W.usable_cells()
print("cells with a natural sound presented on at least 2 trials (Eq. 52 needs m >= 2): %d of %d" % (len(usable), len(cells_w)))
x0, sf0, trig0, notes0 = W.load_trial(next(c for c in cells_w if c["cell"] == usable[0])["files"][0])
print("first file of %s: sampling rate %d Hz, trace %.1f to %.1f mV, notes: %s" % (usable[0], sf0, x0.min(), x0.max(), notes0[:300]))
print("window lengths of the nine models (ms):", [sum(w * k for w, k in zip(W.WIDTHS_MS, c)) for c in W.MODELS],
      "; parameters:", [17 * sum(c) for c in W.MODELS][0])
t = time.time()
STRF = pd.DataFrame([r for rows in A.run_parallel(W.run_cell, [(c, w) for w in (10.0, 20.0) for c in usable]) for r in rows])
print("STRF fits in %.0f s" % (time.time() - t))
print(STRF[(STRF.win_ms == 10) & (STRF.model == 1)][["cell", "n_stimuli", "trials_min", "trials_max", "bins", "peak_lag_ms"]].to_string())
cols = ["strf_lower", "strf_upper", "ln_lower", "ln_upper", "strf_lower_pooled", "strf_upper_pooled", "ln_lower_pooled", "ln_upper_pooled"]
for w in (10.0, 20.0):
    print("Hann window %d ms, means over %d cells:" % (w, STRF[STRF.win_ms == w].cell.nunique()))
    print(STRF[STRF.win_ms == w].groupby("window_ms")[cols].mean().round(3).to_string())
tests = []
for w in (10.0, 20.0):
    d = STRF[STRF.win_ms == w]
    for avg in ["", "_pooled"]:
        for model in ["strf", "ln"]:
            c = model + "_lower" + avg
            a = d[d.model == 9].set_index("cell")[c]; b = d[d.model == 1].set_index("cell")[c]
            fr = stats.friedmanchisquare(*[d[d.model == k].set_index("cell")[c].loc[a.index] for k in range(1, 10)])
            tests.append(dict(window=w, fold_average=("pooled" if avg else "mean of folds"), model=model, cells=len(a),
                              lower_180ms=b.mean(), lower_4052ms=a.mean(), cells_improved=int((a > b).sum()),
                              wilcoxon_p_one_sided=stats.wilcoxon(a, b, alternative="greater").pvalue, friedman_p=fr.pvalue,
                              max_mean_lower=d.groupby("model")[c].mean().max(),
                              max_mean_upper=d.groupby("model")[c.replace("lower", "upper")].mean().max()))
tests = pd.DataFrame(tests)
tests["lower_at_most_0.22"] = tests.max_mean_lower <= 0.22
tests["upper_at_most_0.22"] = tests.max_mean_upper <= 0.22
tests["no_improvement"] = tests.wilcoxon_p_one_sided >= 0.05
print(tests.round(4).to_string())
print("folds whose ridge value is at the low or high end of the grid: %d and %d of %d"
      % (STRF.folds_ridge_at_low_edge.sum(), STRF.folds_ridge_at_high_edge.sum(), 10 * len(STRF)))
print("validation performance of single folds ranges from %.2f to %.2f" % (STRF.fold_valid_min.min(), STRF.fold_valid_max.max()))
fig, ax = plt.subplots(figsize=(6, 3))
for c, ls in [("strf_lower", "-"), ("strf_upper", "--"), ("ln_lower", ":"), ("ln_upper", "-.")]:
    m_ = STRF[STRF.win_ms == 10].groupby("window_ms")[c].mean()
    ax.plot(m_.index / 1000, m_.values, "k" + ls, label=c)
ax.set_xlabel("window length (s)"); ax.set_ylabel("model performance"); ax.set_ylim(-0.5, 0.5); ax.legend(fontsize=7)
pm = STRF[STRF.win_ms == 10].groupby("window_ms")[cols[:4]].mean()
print("plotted means (10-ms window) range %.3f to %.3f; %d of %d outside the axes (-0.5 to 0.5)"
      % (pm.min().min(), pm.max().max(), int(((pm < -0.5) | (pm > 0.5)).sum().sum()), pm.size))

**Result: not reproduced, resting on the criterion's test; performance stays low, but extending the window improves it significantly.** Only 16 of the 25 cells have natural sounds presented on at least two trials, so the test is over 16 cells, not 20. The mean validation (lower) performance of the STRF is 0.167 at 180 ms and 0.207 at 4052 ms with the 10-ms window (0.157 and 0.198 with 20 ms), at most 0.22 at every window when folds are averaged as the Methods do; pooled over folds it reaches 0.244. The training (upper) estimate reaches 0.286. The LN validation mean is 0.05 to 0.14 averaged over folds, below the STRF's, and 0.18 to 0.24 pooled; its training estimate is 0.02 to 0.04 above the STRF's. The one-sided Wilcoxon test finds a higher validation performance at 4052 ms than at 180 ms in every reading, for the STRF (13 to 16 of 16 cells improve, p <= 0.0005) and for LN (p <= 0.0065), so the claim that the longer window did not improve the models does not hold here. Most of the gain is reached by 0.5 to 0.9 s. The Wehr traces are sampled at 4 kHz and recorded with QX-314 in the pipette, as printed.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay
> within about 100 ms, much faster than the second-long timescale of context dependence in A1.

Noise is each trial less its context mean (Eq. 14); spontaneous activity is the intersequence
silences (whole, or without their first 1 s), median filtered as for the responses. The
autocorrelation is normalised by the variance (Eq. 45) and averaged.

In [ ]:
def lag(ac, th):
    i = np.where(ac < th)[0]
    return i[0] * 1e-3 if len(i) else np.nan
nac = np.mean([p["noise_ac"] for f, p in nat_probes()], axis=0)
sp = [f["spont_ac"] for f in NAT if f["spont_ac"] is not None]
sp1 = [f["spont_ac_skip1s"] for f in NAT if f["spont_ac_skip1s"] is not None]
sac, sac1 = np.mean(sp, axis=0), np.mean(sp1, axis=0)
tl = []
for name, ac, n in [("noise", nac, len(nat_probes())), ("spontaneous", sac, len(sp)), ("spontaneous, first 1 s left out", sac1, len(sp1))]:
    tl.append(dict(series=name, n=n, lag_1_over_e_s=lag(ac, 1 / np.e), lag_0_1_s=lag(ac, 0.1)))
tl = pd.DataFrame(tl)
print(tl.to_string())
per_file = [lag(np.mean([p["noise_ac"] for p in f["probes"] if "noise_ac" in p and p["m"] >= 4], axis=0), 1 / np.e) for f in NAT]
print("noise 1/e lag per file: median %.3f s, range %.3f to %.3f s" % (np.median(per_file), np.min(per_file), np.max(per_file)))
print("all 1/e and 0.1 lags under 150 ms:", bool((tl[["lag_1_over_e_s", "lag_0_1_s"]] < 0.15).all().all()),
      "; 1/e lags below 1.04/3 s:", bool((tl.lag_1_over_e_s < 1.04 / 3).all()))

**Result: reproduced; both autocorrelations decay within about 0.1 s, much faster than the second-long context dependence.** The noise autocorrelation falls below 1/e at 0.07 s and below 0.1 at 0.11 s. The spontaneous autocorrelation does the same at 0.08 and 0.12 s. All lags are under 150 ms and far below a third of 1.04 s.

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the intersequence
> silences, but the response power attributable to the preceding stimulus alone faded within about
> 1 s after the stimulus ended.

The silence probe is the first 2.5 s of each intersequence silence (`CxtDep.m` default, cut to the
shortest silence), with the last fragment of each sequence as its context; at least 4 trials per
context.

In [ ]:
SP = [(f, p) for f in NAT for p in f["probes"] if p["probe"] == -1 and p["m"] >= 4]
starts = [(A.cell_id(f["path"]), p["all"]["runs"][0][0] / 1e4) for f, p in SP if len(p["all"]["runs"])]
early_cells = sorted({c for c, s0 in starts if s0 < 0.5})
print("silence probes: %d in %d cells; with a significant period: %d; cells with one starting within 0.5 s: %d"
      % (len(SP), len({A.cell_id(f["path"]) for f, p in SP}), len(starts), len(early_cells)))
L = min(len(p["all"]["pred"]) for f, p in SP)
t = np.arange(L) * 1e-3
pr = np.mean([p["all"]["pred"][:L] for f, p in SP], axis=0)
cd = np.mean([p["all"]["pred"][:L] - p["all"]["inv"][:L] for f, p in SP], axis=0)
for name, y in [("stimulus-related power (Eq. 17)", pr), ("context-dependent power (Eq. 19)", cd)]:
    y10 = A.bin_mean(y, 10)
    t10 = np.arange(len(y10)) * 0.01
    below = np.where(y10 < 0.1 * y10[0])[0]
    print("%s: %.2f mV^2 at onset (first 10 ms), %.3f at 1 s, %.3f at 2 s; first below 10%% of onset at %s s"
          % (name, y10[0], y10[100], y10[200], round(t10[below[0]], 2) if len(below) else "never"))
print("silence probe length used: %.2f s" % (L / 1000))

In [ ]:
# chance level for the early start: the fraction of shuffles (trials shuffled across contexts)
# with a significant point in the first 0.5 s, per probe, and the expected number of cells
nf_sp = np.mean([p["all"]["noise"][:500].mean() for f, p in SP])
pe = collections.defaultdict(list)
for f, p in SP:
    pe[A.cell_id(f["path"])].append(p["all"]["noise_early"])
exp_cells = sum(1 - np.prod(1 - np.array(v)) for v in pe.values())
print("silence probes: mean noise floor over 0 to 0.5 s %.4f; fraction of shuffles with a significant point in"
      " the first 0.5 s, median over probes %.3f (range %.3f to %.3f)"
      % (nf_sp, np.median([p["all"]["noise_early"] for f, p in SP]), min(p["all"]["noise_early"] for f, p in SP),
         max(p["all"]["noise_early"] for f, p in SP)))
print("cells expected by chance to have a significant period starting within 0.5 s: %.1f of %d (observed %d)"
      % (exp_cells, len(pe), len(early_cells)))

**Result: reproduced; 19 cells show significance soon after the silence begins, and the power fades within 1 s.** There are 44 silence probes in 38 cells. 19 cells have a significant period starting within 0.5 s of the silence onset, where shuffling the trials across contexts gives 3.2 cells by chance (noise floor 0.0054 over the first 0.5 s). The stimulus-related power over silence probes (Eq. 17) falls below 10% of its onset value at 0.72 s, and the context-dependent power (Eq. 19) at 0.46 s, both before 1.1 s.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short time
> (time constant about 80 ms; 93 probes in 14 cells), so the long-lasting component in A1 arises
> mainly in cortex.

PSTHs in 100-ms bins for k = 0 to 60, SD over contexts (n - 1), averaged over the 93 probes; one
exponential with a free constant. Readings: bin time at its start or centre; spike threshold; the
average over all probes or over probes running.

In [ ]:
MP = [(g, p) for g in MGB for p in g["probes"] if p["m"] >= 10]
out = []
for key in A.MGB_THRESHOLDS:
    sds = [p["psth_%s" % key].std(axis=0, ddof=1)[:61] for g, p in MP]
    S, R = np.zeros(61), np.zeros(61)
    for x in sds:
        S[:len(x)] += x; R[:len(x)] += 1
    for denom in ["all", "running"]:
        y = S / (len(MP) if denom == "all" else np.maximum(R, 1))
        for tb, off in [("start", 0.0), ("centre", 0.05)]:
            tt = np.arange(61) * 0.1 + off
            f = A.fit_exp(tt, y, 1, None)
            out.append(dict(threshold=key, denominator=denom, bin_time=tb, a1=f["a"][0], tau1=f["tau"][0],
                            kappa=f["kappa"], mean_after_0_5s=y[tt > 0.5].mean(), first_bins=np.round(y[:3], 2)))
out = pd.DataFrame(out)
out["tau_ok"] = ((out.tau1 - 0.08).abs() <= 0.008) | (out.tau1 < 0.1)
out["plateau_vs_fit"] = (out.mean_after_0_5s - out.kappa).abs() <= 0.25 * out.kappa.abs()
out["plateau_vs_2"] = (out.mean_after_0_5s - 2.0).abs() <= 0.5
print(out.round(3).to_string())
print("probes per cell:", dict(collections.Counter(A.cell_id(g["path"]) for g, p in MP)))

In [ ]:
# the same without the two cells whose trigger alignment the clock check did not establish
bad = {"20071012-ha1-002", "20071012-ha1-003"}
MPg = [(g, p) for g, p in MP if A.cell_id(g["path"]) not in bad]
outg = []
for key in A.MGB_THRESHOLDS:
    sds = [p["psth_%s" % key].std(axis=0, ddof=1)[:61] for g, p in MPg]
    S, R = np.zeros(61), np.zeros(61)
    for x in sds:
        S[:len(x)] += x; R[:len(x)] += 1
    for denom in ["all", "running"]:
        y = S / (len(MPg) if denom == "all" else np.maximum(R, 1))
        tt = np.arange(61) * 0.1
        f = A.fit_exp(tt, y, 1, None)
        outg.append(dict(threshold=key, denominator=denom, a1=f["a"][0], tau1=f["tau"][0], kappa=f["kappa"],
                         mean_after_0_5s=y[tt > 0.5].mean()))
outg = pd.DataFrame(outg)
outg["tau_ok"] = ((outg.tau1 - 0.08).abs() <= 0.008) | (outg.tau1 < 0.1)
outg["plateau_vs_fit"] = (outg.mean_after_0_5s - outg.kappa).abs() <= 0.25 * outg.kappa.abs()
print("without cells %s: %d probes in %d cells (bins timed at their start)" % (sorted(bad), len(MPg), len({A.cell_id(g["path"]) for g, p in MPg})))
print(outg.round(3).to_string())

**Result: reproduced; the time constant is 0.06 to 0.08 s in every reading, and the SD stays flat after 0.5 s.** Under every reading the time constant is below 0.1 s, and 0.080 to 0.083 s when the average is over all probes. The mean SD after 0.5 s is within 25% of the fitted constant in every reading, and within 25% of the paper's 2.0 spike/s in two. Without the two cells whose clock check failed (78 probes in 12 cells), the time constants are 0.057 to 0.084 s and the plateau test still holds in every reading. The sizes depend on the spike threshold and the bin time (5.2 to 20.8 spike/s). One is 15.0 (40 SD threshold, bin centres, all probes), but its constant is 1.0 spike/s, not the paper's 2.0.

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first 100-ms
> bin of the PSTH.

Among MGB probes from the seven-fragment design of Figure 2 with three contexts (silence and two
fragments), at least one with the SD over contexts in the first bin at least twice that of every
later bin; the fraction of the 93 probes for which this holds is also given.

In [ ]:
def first_bin(p, key):
    s = p["psth_%s" % key].std(axis=0, ddof=1)
    return s[0] >= 2 * s[1:].max() and s[0] > 0
for key in A.MGB_THRESHOLDS:
    allp = [first_bin(p, key) for g, p in MP]
    des = [(A.cell_id(g["path"]), p["probe"], first_bin(p, key)) for g, p in MP
           if g["n_frag"] == 7 and len(p["contexts"]) == 3 and 0 in p["contexts"]]
    print("threshold %s: %d of %d probes; in the Figure 2 design %d of %d: %s"
          % (key, sum(allp), len(allp), sum(d[2] for d in des), len(des), [d[:2] for d in des if d[2]]))

**Result: reproduced; 4 of the 12 probes in the Figure 2 design have the first-bin pattern under every threshold, and so do 35 to 46 of the 93 probes.** All four come from cell 20071010-ha2-003.

## fig7-example-frequency-context

> In an example A1 neuron, a natural sound probe preceded by three DMR contexts that differed only
> in frequency band gave significant context dependence for about 4 s.

The criterion asks for a DMR frequency-variant cell with contexts in 0.625-2.5, 2.5-10 and 10-40 kHz
and a natural sound probe, and says the claim is not testable when no cell matches. The frequency
bands (`ff`) of every DMR and TORC stimulus in the synthetic files are listed here.

In [ ]:
import re
bands = []
for f in SYNR:
    for k, a in f["attrs"].items():
        if a["kind"] in ("DMR", "TORC") and "ff" in a["par"]:
            bands.append(dict(file=A.cell_id(f["path"]) + "/" + f["path"][-7:-4], kind=a["kind"], stimulus=k,
                              band=tuple(int(v) for v in re.findall(r"\d+", a["par"]["ff"]))))
bands = pd.DataFrame(bands)
print(bands.groupby(["file", "kind"]).band.agg(lambda b: sorted(set(b))).to_string())
target = {(625, 2500), (2500, 10000), (10000, 40000)}
match = bands.groupby("file").band.agg(lambda b: target <= set(b))
print("files with DMR or TORC contexts in all three bands of Figure 7 (0.625-2.5, 2.5-10, 10-40 kHz):", int(match.sum()))
fp = ok4[(ok4.row == "Natural") & (ok4.prop == "frequency") & (ok4.family == "DMR")]
late = []
for f in SYNR:
    for p in f["probes"]:
        if "res" in p and p["kind"] == "natural" and p["property"] == "frequency" and ens_family(f, p) == "DMR":
            late.append(len(p["res"]["runs"]) > 0 and p["res"]["runs"][-1][1] / 1e4 >= 3.6)
print("weaker check, not the claim: natural probes after DMR frequency variants: %d in %d cells; with a significant"
      " period ending at or after 3.6 s: %d" % (len(fp), fp.cell.nunique(), sum(late)))

**Result: blocked; no cell in the data set has the example's design.** No DMR or TORC file has contexts in the three bands of Figure 7 (0.625-2.5, 2.5-10 and 10-40 kHz). The DMR files with a natural sound probe use two-octave bands from 0.25-1 to 8-32 kHz, and the DMR bandwidth files use 0.625-10, 1.25-20, 2.5-40 and 0.625-2.5 plus 10-40 kHz without a natural probe. As a weaker check that is not the claim, 1 of the 5 natural probes after DMR frequency variants has a significant period ending at or after 3.6 s.

## fig8a-proportion-by-property

> When only one property of synthetic conditioning stimuli varied, context dependence was found in
> 77 of 95 probes (81.1%, 31 cells) for amplitude, 73 of 110 (35 cells) for frequency, 62 of 96
> (64.6%, 27 cells) for AM, 47 of 82 (57.3%, 25 cells) for FM and 59 of 137 (43.1%, 27 cells) for
> higher-order properties.

For each probe of the synthetic files, its contexts are restricted to the variants of one property
(Methods, Stimulus design, paragraph 2: "the analysis should be conducted by using only those
frequency variants ... as conditioning stimuli"), so silence and unrelated fragments are left out.
The property between two stimuli is read from their parameters (`A.PROPERTY_OF`; the notes of the
files of a natural probe after modulated harmonic tones name which parameters make amplitude,
frequency, AM and FM variants), and a natural sound against the pink noise made from it is a
higher-order variant. A probe tested with several sets of variants of one property (the 4 x 4 grids
of AM and FM rates) is counted once, with the largest set and, among equal sets, the one with the
lowest stimulus numbers (assumption). Each set needs at least 4 trials in each of at least 2
contexts (Table 1 legend). Natural and synthetic probes are combined, as the Results do. The Table 1
counts are compared under methods-recording-counts above.

In [ ]:
EP = [(f, p) for f in SYNR for p in f["probes"] if "res" in p]
exp8a = {"amplitude": (77, 95), "frequency": (73, 110), "AM": (62, 96), "FM": (47, 82), "higher-order": (59, 137)}
r8 = []
for P in A.PROPERTIES:
    d = ok4[ok4.prop == P]
    r8.append(dict(property=P, probes=len(d), cells=d.cell.nunique(), animals=d.animal.nunique(), significant=int(d.sig.sum()),
                   fraction=d.sig.mean(), paper_significant=exp8a[P][0], paper_probes=exp8a[P][1],
                   paper_fraction=exp8a[P][0] / exp8a[P][1], count_ok=abs(d.sig.sum() - exp8a[P][0]) <= 0.1 * exp8a[P][0]))
r8 = pd.DataFrame(r8).set_index("property")
print(r8.round(3).to_string())
fr = r8.fraction
print("order amplitude > frequency: %s; AM > FM > higher-order: %s" % (fr["amplitude"] > fr["frequency"],
      fr["AM"] > fr["FM"] > fr["higher-order"]))
print("by probe type:")
print(ok4.groupby(["prop", "row"]).sig.agg(["sum", "size", "mean"]).round(3).to_string())
d0 = ok4[~ok4.description.str.contains("naturalistic")]
print("without the files of 'naturalistic' sounds (no Table 1 column named for them):",
      {P: "%d/%d" % (d0[d0.prop == P].sig.sum(), (d0.prop == P).sum()) for P in A.PROPERTIES})
pa = ok4.groupby(["prop", "animal"]).sig.mean().groupby("prop").agg(["size", "median", "min", "max"])
print("fraction significant per animal:"); print(pa.round(2).to_string())

**Result: partially reproduced; every count is below the paper's, but the proportions keep the paper's order.** The probes tested number 95 for amplitude and 110 for frequency, the paper's totals, but 39 for AM and 25 for FM against 96 and 82 (the modulated harmonic tone probes are missing, see methods-recording-counts) and 131 for higher-order against 137. 57, 63, 18, 10 and 45 are significant, against 77, 73, 62, 47 and 59; none is within 10%. The fractions are 0.600, 0.573, 0.462, 0.400 and 0.344, so amplitude > frequency and AM > FM > higher-order, the partial rule. All fractions are below the paper's (0.811, 0.664, 0.646, 0.573, 0.431), as the natural sound fraction is (0.61 against 0.669). Without the files of naturalistic sounds, amplitude and frequency give 40/65 and 50/80.

## fig8a-significance-fits and fig8b-decay-by-property

> fig8a-significance-fits: The proportion of probes with significant context dependence decays
> fastest and is smallest for higher-order changes, with fitted (size, time constant) of (0.23,
> 0.25 s) for amplitude, (0.19, 0.27 s) for frequency, (0.13, 0.17 s) for AM, (0.07, 0.15 s) for FM
> and (0.04, 0.13 s) for higher-order properties.

> fig8b-decay-by-property: The normalized context-dependent power decays slowly for amplitude and
> frequency changes (time constants 0.66 and 0.97 s) and fast for AM, FM and higher-order changes
> (0.11, 0.15 and 0.20 s), so higher-order properties act mainly on a timescale of about 100 ms.

Over the probes of fig8a-proportion-by-property, per property: the proportion of probes significant
at each time, with one exponential and the constant fixed at the mean noise floor (1000 shuffles per
probe); and Q(t) of Eq. 20 (denominator smoothed over [0, 2t]) with one exponential and constant 0.
Both are fitted from 0 to 4 s. Every synthetic probe lasts at least 4 s, so the proportion is over
all probes at every time, and the variant of fig8b (0 to the shortest probe length) is the same
range.

In [ ]:
L8 = 4000  # 0 to 4 s, the plotted range; every synthetic probe lasts at least 4 s
t8 = np.arange(L8) * 1e-3
print("shortest synthetic probe: %.2f s" % (min(p["T"] for f, p in EP) / 1e4))
CURVES = {}
fit8 = []
for P in A.PROPERTIES:
    PP = [p for f, p in EP if p["property"] == P]
    sig = np.zeros(L8); nf = np.zeros(L8); PRm = []; IVm = []
    for p in PP:
        s = np.zeros(p["T"], bool)
        for a, b in p["res"]["runs"]:
            s[a:b] = True
        sig += A.bin_mean(s.astype(float), 10)[:L8]
        nf += p["res"]["noise"][:L8]
        PRm.append(p["res"]["pred"][:L8]); IVm.append(p["res"]["inv"][:L8])
    sig /= len(PP); nf /= len(PP)
    PRm, IVm = np.array(PRm, float), np.array(IVm, float)
    pr, iv = PRm.mean(0), IVm.mean(0)
    Q8 = (pr - iv) / A.running_mean_0_2t(pr, 1e-3)
    CURVES[P] = dict(sig=sig, nf=nf, PR=PRm, IV=IVm, Q=Q8)
    kap = nf.mean()
    fs = A.fit_exp(t8, sig, 1, kappa=kap)
    fq = A.fit_exp(t8, Q8, 1, kappa=0.0)
    fit8.append(dict(property=P, probes=len(PP), kappa=kap, sig_a1=fs["a"][0], sig_tau1=fs["tau"][0],
                     Q_a1=fq["a"][0], Q_tau1=fq["tau"][0], Q_at_0=Q8[0], Q_at_1s=Q8[1000], Q_min=Q8.min(), Q_max=Q8.max()))
fit8 = pd.DataFrame(fit8).set_index("property")
e_sig = {"amplitude": (0.23, 0.25, 2.8e-3), "frequency": (0.19, 0.27, 2.3e-3), "AM": (0.13, 0.17, 2.3e-3),
         "FM": (0.07, 0.15, 2.1e-3), "higher-order": (0.04, 0.13, 3.6e-3)}
e_Q = {"amplitude": (0.57, 0.66), "frequency": (0.54, 0.97), "AM": (0.86, 0.11), "FM": (0.64, 0.15), "higher-order": (0.52, 0.20)}
for P in A.PROPERTIES:
    fit8.loc[P, "paper_sig_a1"], fit8.loc[P, "paper_sig_tau1"], fit8.loc[P, "paper_kappa"] = e_sig[P]
    fit8.loc[P, "paper_Q_a1"], fit8.loc[P, "paper_Q_tau1"] = e_Q[P]
fit8["sig_a1_ok"] = (fit8.sig_a1 - fit8.paper_sig_a1).abs() <= 0.1 * fit8.paper_sig_a1 + 1e-12
fit8["sig_tau1_ok"] = (fit8.sig_tau1 - fit8.paper_sig_tau1).abs() <= 0.1 * fit8.paper_sig_tau1 + 1e-12
fit8["Q_tau1_ok"] = (fit8.Q_tau1 - fit8.paper_Q_tau1).abs() <= 0.1 * fit8.paper_Q_tau1 + 1e-12
fit8["Q_a1_ok"] = (fit8.Q_a1 - fit8.paper_Q_a1).abs() <= 0.1 * fit8.paper_Q_a1 + 1e-12
print(fit8.round(4).T.to_string())
a1 = fit8.sig_a1
print("significance sizes in the printed order amplitude > frequency > AM > FM > higher-order:",
      bool(a1["amplitude"] > a1["frequency"] > a1["AM"] > a1["FM"] > a1["higher-order"]))
tq = fit8.Q_tau1
print("Q constants: amplitude and frequency above 0.4 s: %s; AM, FM, higher-order below 0.3 s: %s"
      % (bool((tq[["amplitude", "frequency"]] > 0.4).all()), bool((tq[["AM", "FM", "higher-order"]] < 0.3).all())))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for P in A.PROPERTIES:
    ax[0].plot(t8, CURVES[P]["sig"], lw=0.8, label=P)
    ax[1].plot(t8, CURVES[P]["Q"], lw=0.8, label=P)
ax[0].set_title("proportion significant"); ax[1].set_title("Q(t)"); ax[1].set_ylim(-0.5, 1.5)
for a_ in ax: a_.set_xlabel("time from probe onset (s)")
ax[0].legend(fontsize=7)
print("points of Q outside -0.5 to 1.5:", {P: int(((CURVES[P]["Q"] < -0.5) | (CURVES[P]["Q"] > 1.5)).sum()) for P in A.PROPERTIES})

**Result: fig8a-significance-fits is not reproduced; three of the ten values are within tolerance and the sizes are not in the printed order.** The time constants are 0.280, 0.286, 0.176, 0.133 and 0.129 s (amplitude, frequency, AM, FM, higher-order), within tolerance for frequency, AM and higher-order. The sizes are 0.123, 0.147, 0.086, 0.036 and 0.026, all below the paper's 0.23, 0.19, 0.13, 0.07 and 0.04, and frequency is larger than amplitude, so the order of the partial rule fails. The noise floors are 3.3, 2.3, 2.2, 2.1 and 2.4 x 10^-3 against 2.8, 2.3, 2.3, 2.1 and 3.6 x 10^-3.

**Result: fig8b-decay-by-property is partially reproduced; slow decay for amplitude and frequency and fast decay for AM, FM and higher-order, but no constant is within tolerance.** The time constants of Q(t) are 1.34 and 0.85 s for amplitude and frequency (paper 0.66 and 0.97 s), both above 0.4 s, and 0.095, 0.113 and 0.176 s for AM, FM and higher-order (paper 0.11, 0.15 and 0.20 s), all below 0.3 s. The sizes are 0.40, 0.49, 0.99, 0.71 and 1.23 (paper 0.57, 0.54, 0.86, 0.64, 0.52); the higher-order Q starts at 1.58 and exceeds 1.5 at 14 points.

## fig8c-relative-effects

> Changes in amplitude and frequency of the conditioning stimuli produced context effects as large
> and long-lasting as natural sound contexts, while AM, FM and higher-order changes produced
> substantially smaller and shorter effects; A1 is more sensitive to lower-order than to higher-order
> properties.

Per the Methods (Eq. 21 and the paragraph after it), the area under Q(t) from 0 to 4 s for each
property divided by the area for natural sounds (Figure 3B: 305 probes, silence counted), with 95%
intervals from 200 resamplings of 1000 probes drawn with replacement (the probe reading). The
reading that resamples trials within each probe and context was not run.

In [ ]:
# area under Q from 0 to 4 s relative to natural sounds (Figure 3B: silence counted, all probes)
PPn = nat_probes("all")
PRn = np.array([np.pad(p["all"]["pred"][:L8], (0, max(0, L8 - len(p["all"]["pred"])))) for f, p in PPn], float)
IVn = np.array([np.pad(p["all"]["inv"][:L8], (0, max(0, L8 - len(p["all"]["inv"])))) for f, p in PPn], float)
def q_area(PR, IV, w=None):
    pr = PR.mean(0) if w is None else w @ PR / w.sum()
    iv = IV.mean(0) if w is None else w @ IV / w.sum()
    return ((pr - iv) / A.running_mean_0_2t(pr, 1e-3)).sum() * 1e-3
rng8 = np.random.default_rng(20091108)
an = q_area(PRn, IVn)
nat_boot = np.array([q_area(PRn, IVn, rng8.multinomial(1000, np.ones(len(PRn)) / len(PRn)).astype(float)) for _ in range(200)])
rel = []
for P in A.PROPERTIES:
    c = CURVES[P]
    a = q_area(c["PR"], c["IV"])
    boot = np.array([q_area(c["PR"], c["IV"], rng8.multinomial(1000, np.ones(len(c["PR"])) / len(c["PR"])).astype(float))
                     for _ in range(200)]) / nat_boot
    lo, hi = np.percentile(boot, [2.5, 97.5])
    rel.append(dict(property=P, area=a, natural_area=an, relative=a / an, ci_low=lo, ci_high=hi))
rel = pd.DataFrame(rel).set_index("property")
print("natural sound area under Q(t), 0 to 4 s: %.3f s (%d probes)" % (an, len(PRn)))
print(rel.round(3).to_string())
low_lb = min(rel.loc["amplitude", "ci_low"], rel.loc["frequency", "ci_low"])
ho = rel.loc[["AM", "FM", "higher-order"]]
print("amplitude and frequency intervals include 1 or lie above it: %s" % bool((rel.loc[["amplitude", "frequency"], "ci_high"] >= 1).all()))
print("AM, FM, higher-order intervals below 1 and below the lower bounds of amplitude and frequency: %s"
      % bool(((ho.ci_high < 1) & (ho.ci_high < low_lb)).all()))
pt = rel.relative
print("point estimates: amplitude and frequency at least 1: %s; AM, FM, higher-order below 1 and below both: %s"
      % (bool((pt[["amplitude", "frequency"]] >= 1).all()),
         bool((pt[["AM", "FM", "higher-order"]] < 1).all() and min(pt["amplitude"], pt["frequency"]) > max(pt[["AM", "FM", "higher-order"]]))))

**Result: not reproduced; amplitude and frequency effects are smaller than those of natural sounds, though larger than those of AM, FM and higher-order changes.** Relative to natural sounds (area 0.734 s), the area under Q(t) is 0.74 for amplitude (95% interval 0.65 to 0.86) and 0.67 for frequency (0.58 to 0.77), so neither interval includes 1. AM, FM and higher-order give 0.09, 0.20 and 0.51, all below 1 and below amplitude and frequency; the higher-order interval (0.44 to 0.61) overlaps the lower bound for frequency. As the point estimates of amplitude and frequency are also below 1, the partial rule is not met.

## fig8-natural-synthetic-probes-alike

> Natural and synthetic probes gave no difference in the context effects of the synthetic
> conditioning stimuli, so they were combined.

The test fixed by the criterion: per property, Fisher's exact test of the proportion of significant
probes between natural and synthetic probes, at 0.05 with Bonferroni correction over the properties
tested with both probe types. The test is not the paper's, so the outcome rests on it.

In [ ]:
from scipy import stats
ns = []
for P in A.PROPERTIES:
    d = ok4[ok4.prop == P]
    nat = d[d.row == "Natural"]; syn = d[d.row != "Natural"]
    if len(nat) and len(syn):
        tab = [[int(nat.sig.sum()), int((~nat.sig).sum())], [int(syn.sig.sum()), int((~syn.sig).sum())]]
        o, pv = stats.fisher_exact(tab)
        ns.append(dict(property=P, natural="%d/%d" % (tab[0][0], len(nat)), synthetic="%d/%d" % (tab[1][0], len(syn)),
                       natural_fraction=nat.sig.mean(), synthetic_fraction=syn.sig.mean(), odds_ratio=o, p=pv))
    else:
        print("%s: only %s probes (%d), not tested" % (P, "natural" if len(nat) else "synthetic", len(d)))
ns = pd.DataFrame(ns).set_index("property")
ns["p_bonferroni"] = np.minimum(ns.p * len(ns), 1)
ns["differs"] = ns.p_bonferroni < 0.05
print(ns.round(4).to_string())

**Result: reproduced, resting on the criterion's test; no property shows a difference between natural and synthetic probes.** Four properties have both probe types: amplitude (25/45 natural, 32/50 synthetic), frequency (23/46, 40/64), AM (10/25, 8/14) and higher-order (20/58, 25/73). The odds ratios are 0.70, 0.60, 0.50 and 1.01, and no Bonferroni-corrected P is below 0.05 (smallest 0.97). FM has only natural probes and is not tested.

## Blocked claims

- fig5-example-conductances and fig6-conductance-decay: the data set holds no voltage-clamp
  recordings (data check above), so the conductances cannot be computed.
- fig7-example-frequency-context: no file has the three DMR bands of the example (section above).

In [ ]:
print("notebook ran in %.1f min; peak memory %.2f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

## Summary

| Claim | Outcome |
|---|---|
| methods-recording-counts | partially reproduced (305 probes, 39 cells, 93 MGB probes and 14 MGB cells match; 132 cells against 123; Table 1 matches in part; voltage clamp not in the data) |
| a1-mgb-firing-rates | partially reproduced, on assumed spike detectors (A1 below 1 Hz and MGB evoked above 5 Hz; not all means within 10%) |
| fig2-example-reliability | reproduced (0.31 over cells; the example cell gives 0.61 and 0.74) |
| fig2-example-long-context | reproduced under the amended criterion (no significance from 1.60 to 3.97 s; the strict 2 to 4 s reading fails by 30 ms) |
| fig3a-proportion-significant | reproduced (186 of 305, late fraction 0.252) |
| fig3a-significance-decay | partially reproduced, on the assumption of non-negative sizes |
| fig3b-fractional-power-decay | not reproduced (time constant 1.3 to 1.8 s against 1.04 s) |
| timescale-longer-than-membrane | reproduced |
| fig4-predictability-upper-bound | not reproduced (time constant 1.3 s or longer against 1.04 s) |
| fig4-strf-performance-low | not reproduced, on the criterion's test (performance at most 0.22, but the 4-s window improves on 180 ms; 16 cells) |
| fig5-example-conductances | blocked (data missing) |
| fig6-conductance-decay | blocked (data missing) |
| fig7-example-frequency-context | blocked (data missing: no cell with the example's design) |
| fig8a-proportion-by-property | partially reproduced (counts below the paper's; order of the proportions kept) |
| fig8b-decay-by-property | partially reproduced (slow for amplitude and frequency, fast for the others; no constant within tolerance) |
| fig8a-significance-fits | not reproduced (3 of 10 values within tolerance; sizes not in order) |
| fig8c-relative-effects | not reproduced (amplitude and frequency 0.74 and 0.67 of natural, intervals below 1) |
| fig8-natural-synthetic-probes-alike | reproduced, on the criterion's test |
| fig9-mgb-fast-decay | reproduced |
| fig9-example-first-bin | reproduced |
| figs1-silence-probe | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The Kruskal-Wallis test uses average ranks for ties. The authors' `CxtDep.m` ranks ties by
  order of the trials, which on the quantised traces gives the first contexts lower ranks; ties
  occur at a median of 31% of a probe's time points.
- Details the paper leaves open are taken from the authors' code in the data set: which
  presentations form the trials (the first "Effective repeats" of each sequence), the silence
  probe (first 2.5 s of the intersequence silence), the centring of each trial over the probe
  in the power measure, the PSTH bins and the MGB spike filter.
- A1 spikes are found as excursions above the median-clipped trace, and MGB spikes at three
  thresholds, as the paper gives no detector settings. The rates are over 39 (or 133) A1 and 14
  MGB cells, not the paper's 194 and 16.
- Population curves are formed over all probes and over the probes still running; fits use a
  grid of start values. Bounding the sizes at zero is what lets the two-exponential fit of
  Figure 3A describe the decay; with free signs one exponential fits the rise after onset.
- For the synthetic ensembles, the property varied between two stimuli is read from their
  parameters, and a probe tested with several sets of variants of one property is counted once
  (the largest set, lowest stimulus numbers among equal ones). Reversed or syllable-shuffled
  sounds and other transforms with no Table 1 column are left out as contexts.
- The STRF analysis uses 16 of Wehr's cells, those with natural sounds presented on at least
  two trials, with the response in 4-ms bins and lags before the sound onset set to zero.

What was found about the data set:

- The 42 voltage-clamp recordings (Figures 5 and 6) are not in the data set. Every file is a
  current-clamp or cell-attached trace in mV.
- The data set holds 39 natural sound A1 cells and 14 MGB cells with context files, which give the
  paper's analysed counts exactly (305 and 93 probes). It does not hold the 194 current-clamp
  cells and 16 MGB cells of the firing rates, and 132 A1 cells meet the analysis criterion where
  the paper has 123.
- The synthetic files give the paper's Table 1 totals for amplitude (95 probes) and frequency
  (110), but hold no modulated harmonic tone probes for amplitude, frequency or FM and 14 of the
  71 for AM, so the AM and FM populations of Figure 8 are 39 and 25 probes against 96 and 82.
- No file holds the example of Figure 7 (DMR contexts in 0.625-2.5, 2.5-10 and 10-40 kHz).
- Of Wehr's 25 cells, 9 have each natural sound presented once, which Eq. 52 cannot use, so 16
  cells are available for the STRF analysis where the paper used 20.
- In 20051107-ha1-001 the trigger labels (printed in the data check) do not match the numbering
  of the stimulus table ("Seq.1" to "Seq.4"); triggers are matched to the table by their fragment
  lists.
- For the synthetic ensembles, the property varied among the conditioning stimuli is written in
  the notes of some files ("2-5,8: amplitude variants") and in the stimulus descriptions of others
  ("TORCs: amplitude variants"). For the 4 x 4 designs of modulated harmonic tones and the
  transformed natural sounds it has to be read from the stimulus parameters.
- The high-pass filter of `GetSpikes.m` was designed for 4 kHz sampling (its comment) and is applied
  to 10 kHz traces; its automatic threshold (2/3 of the largest value) gives the lowest MGB rates.
